In [5]:
import torch
import torch.nn as nn
import numpy as np
import random

# =========================
# DATA
# =========================
text = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making.
"""

# =========================
# TOKENIZATION (WORD LEVEL)
# =========================
words = text.lower().split()
vocab = sorted(set(words))

word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for w,i in word_to_idx.items()}

vocab_size = len(vocab)
seq_length = 6

# =========================
# CREATE SEQUENCES
# =========================
X, Y = [], []

for i in range(len(words) - seq_length):
    X.append([word_to_idx[w] for w in words[i:i+seq_length]])
    Y.append(word_to_idx[words[i+seq_length]])

X = torch.tensor(X)
Y = torch.tensor(Y)

# Train/Val split
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
Y_train, Y_val = Y[:split], Y[split:]

# =========================
# MODEL
# =========================
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

model = LSTMModel(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# TRAINING (MINI-BATCH)
# =========================
batch_size = 16
epochs = 80

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(len(X_train))

    total_loss = 0

    for i in range(0, len(X_train), batch_size):
        idx = perm[i:i+batch_size]
        xb = X_train[idx]
        yb = Y_train[idx]

        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    val_out = model(X_val)
    val_loss = loss_fn(val_out, Y_val)

    print(f"Epoch {epoch+1} | Train Loss: {total_loss:.3f} | Val Loss: {val_loss.item():.3f}")

# =========================
# SMART GENERATION
# =========================
def generate(model, seed, length=15, temp=0.7, top_k=5):
    model.eval()
    result = seed.copy()

    for _ in range(length):
        inp = torch.tensor([[word_to_idx[w] for w in result[-seq_length:]]])
        out = model(inp)[0]

        probs = torch.softmax(out / temp, dim=0).detach().numpy()

        # Top-k sampling
        top_idx = np.argsort(probs)[-top_k:]
        probs = probs[top_idx]
        probs = probs / probs.sum()

        next_word = idx_to_word[np.random.choice(top_idx, p=probs)]
        result.append(next_word)

    return " ".join(result)

# =========================
# OUTPUT
# =========================
print("\nGenerated Text:\n")
print(generate(model, ["deep","learning","models","improve","sequence","learning"]))

Epoch 1 | Train Loss: 22.126 | Val Loss: 4.423
Epoch 2 | Train Loss: 21.485 | Val Loss: 4.432
Epoch 3 | Train Loss: 20.993 | Val Loss: 4.442
Epoch 4 | Train Loss: 20.431 | Val Loss: 4.455
Epoch 5 | Train Loss: 19.688 | Val Loss: 4.472
Epoch 6 | Train Loss: 19.145 | Val Loss: 4.495
Epoch 7 | Train Loss: 18.357 | Val Loss: 4.527
Epoch 8 | Train Loss: 17.346 | Val Loss: 4.579
Epoch 9 | Train Loss: 16.120 | Val Loss: 4.664
Epoch 10 | Train Loss: 14.991 | Val Loss: 4.808
Epoch 11 | Train Loss: 13.340 | Val Loss: 5.008
Epoch 12 | Train Loss: 12.170 | Val Loss: 5.224
Epoch 13 | Train Loss: 10.660 | Val Loss: 5.317
Epoch 14 | Train Loss: 9.494 | Val Loss: 5.417
Epoch 15 | Train Loss: 8.115 | Val Loss: 5.604
Epoch 16 | Train Loss: 7.330 | Val Loss: 5.773
Epoch 17 | Train Loss: 6.143 | Val Loss: 5.898
Epoch 18 | Train Loss: 5.555 | Val Loss: 6.007
Epoch 19 | Train Loss: 4.859 | Val Loss: 6.102
Epoch 20 | Train Loss: 4.120 | Val Loss: 6.177
Epoch 21 | Train Loss: 3.764 | Val Loss: 6.230
Epoch 22 

In [6]:
import torch
import torch.nn as nn
import numpy as np
import math

# =========================
# DATA (same as before)
# =========================
text = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making.
"""

# =========================
# TOKENIZATION
# =========================
words = text.lower().split()
vocab = sorted(set(words))

word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for w,i in word_to_idx.items()}

vocab_size = len(vocab)
seq_length = 6

# =========================
# CREATE DATA
# =========================
X, Y = [], []

for i in range(len(words) - seq_length):
    X.append([word_to_idx[w] for w in words[i:i+seq_length]])
    Y.append(word_to_idx[words[i+seq_length]])

X = torch.tensor(X)
Y = torch.tensor(Y)

# =========================
# POSITIONAL ENCODING
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2*i)/d_model)))
                pe[pos, i+1] = math.cos(pos / (10000 ** ((2*i)/d_model)))

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# =========================
# TRANSFORMER MODEL
# =========================
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=0.2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)

        # Causal mask (IMPORTANT)
        seq_len = x.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

        x = self.transformer(x, mask=mask)
        x = x[:, -1, :]

        return self.fc(x)

model = TransformerModel(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================
# TRAINING
# =========================
epochs = 60
batch_size = 16

for epoch in range(epochs):
    perm = torch.randperm(len(X))
    total_loss = 0

    for i in range(0, len(X), batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X[idx], Y[idx]

        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss:.3f}")

# =========================
# GENERATION (SMART)
# =========================
def generate(model, seed, length=15, temp=0.7, top_k=5):
    model.eval()
    result = seed.copy()

    for _ in range(length):
        inp = torch.tensor([[word_to_idx[w] for w in result[-seq_length:]]])
        out = model(inp)[0]

        probs = torch.softmax(out / temp, dim=0).detach().numpy()

        top_idx = np.argsort(probs)[-top_k:]
        probs = probs[top_idx]
        probs = probs / probs.sum()

        next_word = idx_to_word[np.random.choice(top_idx, p=probs)]
        result.append(next_word)

    return " ".join(result)

# =========================
# OUTPUT
# =========================
print("\nGenerated Text:\n")
print(generate(model, ["artificial","intelligence","systems","learn","patterns","from"]))

Epoch 1 | Loss: 33.358
Epoch 2 | Loss: 29.589
Epoch 3 | Loss: 22.750
Epoch 4 | Loss: 16.488
Epoch 5 | Loss: 12.040
Epoch 6 | Loss: 7.911
Epoch 7 | Loss: 5.919
Epoch 8 | Loss: 4.568
Epoch 9 | Loss: 3.410
Epoch 10 | Loss: 2.635
Epoch 11 | Loss: 2.044
Epoch 12 | Loss: 1.633
Epoch 13 | Loss: 1.524
Epoch 14 | Loss: 1.298
Epoch 15 | Loss: 1.064
Epoch 16 | Loss: 0.900
Epoch 17 | Loss: 0.816
Epoch 18 | Loss: 0.656
Epoch 19 | Loss: 0.772
Epoch 20 | Loss: 0.636
Epoch 21 | Loss: 0.557
Epoch 22 | Loss: 0.623
Epoch 23 | Loss: 0.481
Epoch 24 | Loss: 0.444
Epoch 25 | Loss: 0.467
Epoch 26 | Loss: 0.397
Epoch 27 | Loss: 0.340
Epoch 28 | Loss: 0.351
Epoch 29 | Loss: 0.300
Epoch 30 | Loss: 0.285
Epoch 31 | Loss: 0.292
Epoch 32 | Loss: 0.238
Epoch 33 | Loss: 0.246
Epoch 34 | Loss: 0.241
Epoch 35 | Loss: 0.219
Epoch 36 | Loss: 0.200
Epoch 37 | Loss: 0.194
Epoch 38 | Loss: 0.187
Epoch 39 | Loss: 0.193
Epoch 40 | Loss: 0.185
Epoch 41 | Loss: 0.166
Epoch 42 | Loss: 0.162
Epoch 43 | Loss: 0.159
Epoch 44 | Loss